In [2]:
# ============================================================
# NOTEBOOK 08 — AGENTE DE COBRANZA: EDA Y FEATURE ENGINEERING
# ============================================================
# Propósito  : Construir el dataset base para el Early Warning
#              System (EWS) del Agente de Cobranza.
# Fuentes    : installments_payments.csv, POS_CASH_balance.csv,
#              application_train.csv, previous_application.csv
# Output     : data/processed/df_collections.csv
# Autor      : Marín Serrato Barrios
# Fecha      : Mayo 2026
# ============================================================

# ── Librerías ────────────────────────────────────────────────
import pandas as pd       # Manejo de DataFrames y CSV
import numpy as np        # Operaciones matemáticas y arrays
import matplotlib.pyplot as plt   # Visualizaciones base
import seaborn as sns     # Visualizaciones estadísticas
import os                 # Manejo de rutas del sistema
import warnings
warnings.filterwarnings('ignore')  # Silenciar warnings no críticos

# ── Opciones de visualización de pandas ─────────────────────
pd.set_option('display.max_columns', None)   # Mostrar todas las columnas
pd.set_option('display.float_format', '{:.4f}'.format)  # 4 decimales

# ── Rutas del proyecto ───────────────────────────────────────
# os.path.abspath('..') sube un nivel desde notebooks/ hasta la raíz
BASE = os.path.abspath('..')
RAW  = os.path.join(BASE, 'data', 'raw')        # Datos originales sin tocar
PROC = os.path.join(BASE, 'data', 'processed')  # Datos procesados y listos

print('✅ Librerías cargadas')
print(f'   Raíz del proyecto : {BASE}')
print(f'   Datos crudos      : {RAW}')
print(f'   Datos procesados  : {PROC}')

✅ Librerías cargadas
   Raíz del proyecto : C:\Users\Marin\Documents\PROYECTO ML_OPS\credit-risk-scoring-ml
   Datos crudos      : C:\Users\Marin\Documents\PROYECTO ML_OPS\credit-risk-scoring-ml\data\raw
   Datos procesados  : C:\Users\Marin\Documents\PROYECTO ML_OPS\credit-risk-scoring-ml\data\processed


In [16]:
# ============================================================
# CONFIGURACIÓN CENTRALIZADA
# ============================================================
# ⚠️  MODIFICAR AQUÍ — no tocar el código de las celdas siguientes
# Agregar variable  → añadir el nombre a la lista correspondiente
# Quitar variable   → comentar la línea con #
# Cambiar parámetro → modificar el valor directamente
# ============================================================

# ── Variables de application_train.csv ──────────────────────
# Seleccionamos solo las columnas relevantes para cobranza
# para no cargar las 122 columnas del archivo completo en RAM
VARS_APPLICATION = [
    # Identificador único del cliente (llave de unión entre archivos)
    'SK_ID_CURR',

    # Target original: 1 = incumplió, 0 = pagó correctamente
    'TARGET',

    # ── Financieros del crédito ──────────────────────────────
    'AMT_CREDIT',          # Monto total del crédito otorgado
    'AMT_ANNUITY',         # Cuota mensual del crédito
    'AMT_INCOME_TOTAL',    # Ingreso total anual declarado
    'AMT_GOODS_PRICE',     # Precio del bien financiado

    # ── Tiempo ──────────────────────────────────────────────
    'DAYS_BIRTH',          # Días desde el nacimiento (negativo)
    'DAYS_EMPLOYED',       # Días de antigüedad laboral (negativo)
    'DAYS_REGISTRATION',   # Días desde el registro en la institución

    # ── Familia y hogar ──────────────────────────────────────
    'CNT_CHILDREN',        # Número de hijos (proxy carga familiar)
    'CNT_FAM_MEMBERS',     # Total de miembros del hogar

    # ── Activos del cliente ──────────────────────────────────
    'FLAG_OWN_REALTY',     # 1 = tiene propiedad (activo recuperable)
    'FLAG_OWN_CAR',        # 1 = tiene automóvil

    # ── Variables categóricas de perfil ─────────────────────
    'NAME_CONTRACT_TYPE',  # Tipo de crédito: Cash / Revolving
    'NAME_FAMILY_STATUS',  # Estado civil
    'NAME_EDUCATION_TYPE', # Nivel educativo
    'NAME_INCOME_TYPE',    # Tipo de ingreso
    'CODE_GENDER',         # Género

    # ── Geografía ────────────────────────────────────────────
    'REGION_POPULATION_RELATIVE',  # Densidad de la región
    'REGION_RATING_CLIENT',        # Rating de riesgo de la región
]

# ── Variables de previous_application.csv ───────────────────
# Para capturar si el cliente es recurrente o es su primer
# crédito — un cliente recurrente tiene relación establecida
VARS_PREVIOUS = [
    'SK_ID_CURR',           # Llave de unión con el cliente
    'SK_ID_PREV',           # ID de la solicitud anterior
    'NAME_CONTRACT_STATUS', # Estado: Approved/Refused/Canceled
    'AMT_APPLICATION',      # Monto solicitado en apps previas
    'AMT_CREDIT',           # Monto aprobado en apps previas
    'DAYS_DECISION',        # Días desde la decisión previa
]

# ── Parámetros del Early Warning System ─────────────────────
# Umbral de mora: atraso mayor a este número de días
# se considera mora — estándar CNBV para cartera en riesgo
UMBRAL_MORA_DIAS   = 5

# Ventana de predicción: el EWS predice mora en los
# próximos N días — define el horizonte de alerta
VENTANA_PREDICCION = 90

# ── Filtros de historial mínimo para EWS ────────────────────
# Clientes con menos cuotas que estos mínimos se EXCLUYEN
# del entrenamiento — historial insuficiente para construir
# una ventana temporal confiable.
#
# ¿Por qué excluirlos?
# Un cliente con 1-2 cuotas en el futuro casi nunca muestra
# mora — no por ser buen pagador sino por falta de tiempo.
# Eso contaminaría el modelo con falsos negativos.
#
# En producción estos clientes se evalúan con el Agente de
# Originación, no con el EWS.
#
# Referencia de industria: microfinanzas MX usa 3-6 cuotas
# como mínimo para modelos de comportamiento.
MIN_CUOTAS_FUTURAS = 3   # Mínimo de cuotas en período futuro
MIN_CUOTAS_PASADAS = 6   # Mínimo de cuotas en período pasado

# ── Parámetros del modelo ────────────────────────────────────
TEST_SIZE    = 0.20   # 20% para prueba, 80% para entrenamiento
RANDOM_SEED  = 42     # Semilla para reproducibilidad

# ── Parámetros de monitoreo PSI ──────────────────────────────
PSI_UMBRAL_ESTABLE = 0.10   # PSI < 0.10 → modelo estable
PSI_UMBRAL_ALERTA  = 0.25   # PSI > 0.25 → reentrenar modelo

print('✅ Configuración cargada')
print(f'   Variables de application : {len(VARS_APPLICATION)}')
print(f'   Variables de previous    : {len(VARS_PREVIOUS)}')
print(f'   Umbral de mora           : {UMBRAL_MORA_DIAS} días')
print(f'   Ventana de predicción    : {VENTANA_PREDICCION} días')
print(f'   Mín. cuotas futuras      : {MIN_CUOTAS_FUTURAS}')
print(f'   Mín. cuotas pasadas      : {MIN_CUOTAS_PASADAS}')
print(f'   Split train/test         : '
      f'{int((1-TEST_SIZE)*100)}/{int(TEST_SIZE*100)}')

✅ Configuración cargada
   Variables de application : 20
   Variables de previous    : 6
   Umbral de mora           : 5 días
   Ventana de predicción    : 90 días
   Mín. cuotas futuras      : 3
   Mín. cuotas pasadas      : 6
   Split train/test         : 80/20


In [4]:
# ============================================================
# CARGA DE DATOS
# ============================================================
# Cargamos solo las columnas definidas en la configuración
# para optimizar el uso de memoria RAM.
# Los archivos de installments y POS_CASH se cargan completos
# porque todas sus columnas son relevantes para cobranza.
# ============================================================

print("Cargando archivos...")

# ── application_train: datos del solicitante ─────────────────
# usecols=VARS_APPLICATION carga solo las columnas de la config
# Esto reduce el uso de RAM de ~1.5GB a ~200MB
app = pd.read_csv(
    f'{RAW}/application_train.csv',
    usecols=VARS_APPLICATION
)
print(f"  ✅ application_train  : {app.shape[0]:>8,} filas × {app.shape[1]:>3} cols")

# ── installments_payments: historial de pagos por cuota ──────
# Cada fila = una cuota de un crédito
# Es la fuente principal para detectar patrones de atraso
inst = pd.read_csv(f'{RAW}/installments_payments.csv')
print(f"  ✅ installments       : {inst.shape[0]:>8,} filas × {inst.shape[1]:>3} cols")

# ── POS_CASH_balance: estado mensual por producto ────────────
# Cada fila = un mes de un producto crediticio del cliente
# Contiene SK_DPD (Days Past Due) — métrica estándar de mora
pos = pd.read_csv(f'{RAW}/POS_CASH_balance.csv')
print(f"  ✅ POS_CASH_balance   : {pos.shape[0]:>8,} filas × {pos.shape[1]:>3} cols")

# ── previous_application: solicitudes anteriores del cliente ─
# Permite saber si el cliente es nuevo o recurrente
# Un cliente recurrente con buen historial es muy diferente a uno nuevo
prev = pd.read_csv(
    f'{RAW}/previous_application.csv',
    usecols=VARS_PREVIOUS
)
print(f"  ✅ previous_application: {prev.shape[0]:>8,} filas × {prev.shape[1]:>3} cols")

# ── Verificar memoria utilizada ──────────────────────────────
print(f"\n📊 Memoria RAM utilizada:")
for nombre, df in [('app', app), ('inst', inst),
                   ('pos', pos), ('prev', prev)]:
    mb = df.memory_usage(deep=True).sum() / 1024**2
    print(f"   {nombre:<5} : {mb:>7.1f} MB")

Cargando archivos...
  ✅ application_train  :  307,511 filas ×  20 cols
  ✅ installments       : 13,605,401 filas ×   8 cols
  ✅ POS_CASH_balance   : 10,001,358 filas ×   8 cols
  ✅ previous_application: 1,670,214 filas ×   6 cols

📊 Memoria RAM utilizada:
   app   :   164.9 MB
   inst  :   830.4 MB
   pos   :  1137.3 MB
   prev  :   167.1 MB


In [5]:
# ============================================================
# EXPLORACIÓN DE INSTALLMENTS PAYMENTS
# ============================================================
# Antes de calcular features necesitamos entender la estructura
# del archivo: qué significa cada columna, hay nulos, rangos.
# ============================================================

print("=== ESTRUCTURA DEL ARCHIVO ===")
print(inst.dtypes)
print()

print("=== PRIMERAS 5 FILAS ===")
print(inst.head())
print()

print("=== ESTADÍSTICAS DESCRIPTIVAS ===")
print(inst.describe())
print()

print("=== VALORES NULOS ===")
nulos = inst.isnull().sum()
print(nulos[nulos > 0] if nulos.sum() > 0 else "Sin valores nulos")
print()

# ── Verificar la lógica de atraso ────────────────────────────
# DAYS_INSTALMENT: día esperado de pago (negativo = hace N días)
# DAYS_ENTRY_PAYMENT: día real de pago (negativo = hace N días)
# Si DAYS_ENTRY_PAYMENT > DAYS_INSTALMENT → pagó después de la fecha = atraso

diff = inst['DAYS_ENTRY_PAYMENT'] - inst['DAYS_INSTALMENT']
print("=== DISTRIBUCIÓN DE ATRASO (sin clip) ===")
print(f"  Pagos anticipados (diff < 0) : {(diff < 0).sum():>8,} ({(diff<0).mean():.1%})")
print(f"  Pagos a tiempo    (diff = 0) : {(diff == 0).sum():>8,} ({(diff==0).mean():.1%})")
print(f"  Pagos atrasados   (diff > 0) : {(diff > 0).sum():>8,} ({(diff>0).mean():.1%})")
print(f"  Atraso máximo observado      : {diff.max():.0f} días")
print(f"  Atraso promedio (solo tardíos): {diff[diff>0].mean():.1f} días")

=== ESTRUCTURA DEL ARCHIVO ===
SK_ID_PREV                  int64
SK_ID_CURR                  int64
NUM_INSTALMENT_VERSION    float64
NUM_INSTALMENT_NUMBER       int64
DAYS_INSTALMENT           float64
DAYS_ENTRY_PAYMENT        float64
AMT_INSTALMENT            float64
AMT_PAYMENT               float64
dtype: object

=== PRIMERAS 5 FILAS ===
   SK_ID_PREV  SK_ID_CURR  NUM_INSTALMENT_VERSION  NUM_INSTALMENT_NUMBER  \
0     1054186      161674                  1.0000                      6   
1     1330831      151639                  0.0000                     34   
2     2085231      193053                  2.0000                      1   
3     2452527      199697                  1.0000                      3   
4     2714724      167756                  1.0000                      2   

   DAYS_INSTALMENT  DAYS_ENTRY_PAYMENT  AMT_INSTALMENT  AMT_PAYMENT  
0       -1180.0000          -1187.0000       6948.3600    6948.3600  
1       -2156.0000          -2156.0000       1716.5250    17

In [17]:
# ============================================================
# CELDA 5 — FEATURE ENGINEERING: INSTALLMENTS PAYMENTS
# ============================================================
# Objetivo: de 13.6M filas (una por cuota) → una fila por
# cliente con variables que resumen su comportamiento
# histórico de pago.
#
# Este proceso se llama "feature aggregation" — comprimimos
# el historial temporal en estadísticas que el modelo puede
# consumir directamente.
#
# ⚠️  Re-ejecución segura:
# Si cambias UMBRAL_MORA_DIAS en la configuración y
# re-ejecutas esta celda, las columnas anteriores se
# eliminan automáticamente antes de recalcular.
# ============================================================

# ── Limpiar columnas de mora anteriores si existen ───────────
# Necesario cuando se cambia UMBRAL_MORA_DIAS y se re-ejecuta.
# Busca columnas con prefijo 'mora_' y las elimina para evitar
# confusión entre versiones (mora_30d, mora_5d, etc.)
cols_mora_anteriores = [c for c in inst.columns
                        if c.startswith('mora_')]
if cols_mora_anteriores:
    inst.drop(columns=cols_mora_anteriores, inplace=True)
    print(f"♻️  Columnas de mora anteriores eliminadas: "
          f"{cols_mora_anteriores}")
else:
    print("ℹ️  No hay columnas de mora anteriores — primera ejecución")

# ── Calcular días de atraso por cuota ────────────────────────
# Diferencia entre el día real de pago y el día esperado.
# clip(lower=0): si pagó anticipado, atraso = 0
# Un valor positivo significa que pagó después de la fecha
# Un valor de 0 significa que pagó a tiempo o anticipado
inst['dias_atraso'] = (
    inst['DAYS_ENTRY_PAYMENT'] - inst['DAYS_INSTALMENT']
).clip(lower=0)

# ── Indicadores binarios de comportamiento ───────────────────

# pago_parcial: pagó menos del 99% de lo que debía
# El 99% es un margen de tolerancia para diferencias
# de redondeo en los montos
inst['pago_parcial'] = (
    inst['AMT_PAYMENT'] < inst['AMT_INSTALMENT'] * 0.99
).astype(int)

# pago_completo: pagó el 99% o más de lo que debía
inst['pago_completo'] = (
    inst['AMT_PAYMENT'] >= inst['AMT_INSTALMENT'] * 0.99
).astype(int)

# ── Indicador de mora según umbral de configuración ──────────
# Esta columna usa UMBRAL_MORA_DIAS definido en la Celda 2.
# Si cambias el umbral ahí, al re-ejecutar esta celda
# se recalcula automáticamente con el nuevo valor.
# Ejemplo: UMBRAL_MORA_DIAS=5 → crea columna 'mora_5d'
#          UMBRAL_MORA_DIAS=30 → crea columna 'mora_30d'
inst[f'mora_{UMBRAL_MORA_DIAS}d'] = (
    inst['dias_atraso'] > UMBRAL_MORA_DIAS
).astype(int)

print(f"✅ Columnas de comportamiento calculadas:")
print(f"   dias_atraso     : atraso en días por cuota")
print(f"   pago_parcial    : 1 si pagó < 99% del monto")
print(f"   pago_completo   : 1 si pagó ≥ 99% del monto")
print(f"   mora_{UMBRAL_MORA_DIAS}d          : "
      f"1 si atraso > {UMBRAL_MORA_DIAS} días "
      f"(umbral de configuración)")

# ── Agregación por cliente ────────────────────────────────────
# groupby('SK_ID_CURR') agrupa todas las cuotas del mismo
# cliente — puede haber 3 a 277 cuotas por cliente.
# agg() aplica las funciones de resumen a cada grupo.
# El resultado: una fila por cliente con sus estadísticas.
print(f"\nCalculando features de installments "
      f"(puede tardar 1-2 min)...")

features_inst = inst.groupby('SK_ID_CURR').agg(

    # ── Intensidad del atraso ─────────────────────────────
    # ¿Qué tan grave es el peor atraso que ha tenido?
    max_dias_atraso      = ('dias_atraso', 'max'),
    # ¿Cuánto se atrasa en promedio?
    mean_dias_atraso     = ('dias_atraso', 'mean'),
    # ¿Qué tan variable es su comportamiento de atraso?
    # Alta std = comportamiento errático (señal de riesgo)
    std_dias_atraso      = ('dias_atraso', 'std'),

    # ── Frecuencia del atraso ─────────────────────────────
    # ¿Qué porcentaje de sus cuotas las pagó tarde?
    pct_cuotas_atrasadas = ('dias_atraso',
                            lambda x: (x > 0).mean()),
    # ¿Qué porcentaje tuvo atraso grave (30+ días)?
    pct_cuotas_30dias    = ('dias_atraso',
                            lambda x: (x > 30).mean()),
    # ¿Qué porcentaje tuvo atraso muy grave (60+ días)?
    pct_cuotas_60dias    = ('dias_atraso',
                            lambda x: (x > 60).mean()),
    # ¿Qué porcentaje tuvo atraso crítico (90+ días)?
    pct_cuotas_90dias    = ('dias_atraso',
                            lambda x: (x > 90).mean()),

    # ── Comportamiento de pago ────────────────────────────
    # ¿Qué porcentaje de cuotas pagó de forma incompleta?
    # Pagos parciales son señal temprana de dificultad
    pct_pagos_parciales  = ('pago_parcial', 'mean'),
    # ¿Qué porcentaje pagó completo?
    pct_pagos_completos  = ('pago_completo', 'mean'),

    # ── Mora según umbral de configuración ───────────────
    # Cuántas cuotas superaron el umbral definido en config
    n_cuotas_en_mora     = (f'mora_{UMBRAL_MORA_DIAS}d', 'sum'),
    # Proporción de cuotas en mora sobre el total
    pct_cuotas_en_mora   = (f'mora_{UMBRAL_MORA_DIAS}d', 'mean'),

    # ── Volumen del historial ─────────────────────────────
    # ¿Cuántas cuotas tiene en total?
    # Más cuotas = más historial = predicción más confiable
    n_cuotas_total       = ('NUM_INSTALMENT_NUMBER', 'count'),
    # ¿Cuántas cuotas tiene en el último año?
    # Captura si el crédito está activo recientemente
    n_cuotas_recientes   = ('DAYS_INSTALMENT',
                            lambda x: (x > -365).sum()),

).reset_index()

features_inst.columns.name = None

print(f"✅ Features de installments calculadas:")
print(f"   Shape           : {features_inst.shape}")
print(f"   Clientes únicos : {features_inst.shape[0]:,}")
print()
print(features_inst.head())

♻️  Columnas de mora anteriores eliminadas: ['mora_30d']
✅ Columnas de comportamiento calculadas:
   dias_atraso     : atraso en días por cuota
   pago_parcial    : 1 si pagó < 99% del monto
   pago_completo   : 1 si pagó ≥ 99% del monto
   mora_5d          : 1 si atraso > 5 días (umbral de configuración)

Calculando features de installments (puede tardar 1-2 min)...
✅ Features de installments calculadas:
   Shape           : (339587, 14)
   Clientes únicos : 339,587

   SK_ID_CURR  max_dias_atraso  mean_dias_atraso  std_dias_atraso  \
0      100001          11.0000            1.5714           4.1576   
1      100002           0.0000            0.0000           0.0000   
2      100003           0.0000            0.0000           0.0000   
3      100004           0.0000            0.0000           0.0000   
4      100005           1.0000            0.1111           0.3333   

   pct_cuotas_atrasadas  pct_cuotas_30dias  pct_cuotas_60dias  \
0                0.1429             0.0000     

In [18]:
# ============================================================
# FEATURE ENGINEERING — POS_CASH_BALANCE
# ============================================================
# POS_CASH registra el estado MENSUAL de cada producto del cliente.
# SK_DPD = Days Past Due: días de atraso en ese mes.
# Es la métrica estándar que usan los reguladores (CNBV, Basilea).
#
# Diferencia con installments:
# - installments: comportamiento por CUOTA (cada pago individual)
# - POS_CASH: comportamiento por MES (foto mensual del producto)
# Ambas perspectivas son complementarias para el Early Warning.
# ============================================================

print("Calculando features de POS_CASH (puede tardar 1 min)...")

features_pos = pos.groupby('SK_ID_CURR').agg(

    # ── Máximos de DPD ────────────────────────────────────
    # El peor momento del cliente: ¿cuánto llegó a deber en días?
    max_dpd              = ('SK_DPD', 'max'),
    max_dpd_def          = ('SK_DPD_DEF', 'max'),  # Versión regulatoria

    # ── Promedios de DPD ──────────────────────────────────
    # Comportamiento típico: ¿normalmente cuánto se atrasa?
    mean_dpd             = ('SK_DPD', 'mean'),
    mean_dpd_def         = ('SK_DPD_DEF', 'mean'),

    # ── Frecuencia de mora ────────────────────────────────
    # ¿Qué porcentaje de meses tuvo algún atraso?
    pct_meses_dpd        = ('SK_DPD',
                            lambda x: (x > 0).mean()),

    # ¿Qué porcentaje de meses tuvo mora grave (30+ días)?
    pct_meses_dpd30      = ('SK_DPD',
                            lambda x: (x >= UMBRAL_MORA_DIAS).mean()),

    # ── Antigüedad del producto ───────────────────────────
    # ¿Cuántos meses lleva activo el crédito?
    n_meses_activo       = ('MONTHS_BALANCE', 'count'),

).reset_index()

features_pos.columns.name = None

print(f"✅ Features de POS_CASH: {features_pos.shape}")
print(f"   Clientes únicos procesados: {features_pos.shape[0]:,}")
print()
print(features_pos.head())

Calculando features de POS_CASH (puede tardar 1 min)...
✅ Features de POS_CASH: (337252, 8)
   Clientes únicos procesados: 337,252

   SK_ID_CURR  max_dpd  max_dpd_def  mean_dpd  mean_dpd_def  pct_meses_dpd  \
0      100001        7            7    0.7778        0.7778         0.1111   
1      100002        0            0    0.0000        0.0000         0.0000   
2      100003        0            0    0.0000        0.0000         0.0000   
3      100004        0            0    0.0000        0.0000         0.0000   
4      100005        0            0    0.0000        0.0000         0.0000   

   pct_meses_dpd30  n_meses_activo  
0           0.1111               9  
1           0.0000              19  
2           0.0000              28  
3           0.0000               4  
4           0.0000              11  


In [19]:
# ============================================================
# FEATURE ENGINEERING — PREVIOUS APPLICATION
# ============================================================
# Un cliente recurrente (con créditos anteriores) es muy diferente
# a uno nuevo para efectos de cobranza:
# - El recurrente ya tiene relación con la institución
# - Se puede analizar su comportamiento histórico de pago
# - Tiene estrategia de contacto ya establecida
#
# Este archivo tiene 1.67M de filas porque un cliente puede
# haber tenido múltiples solicitudes previas.
# ============================================================

print("Calculando features de previous_application...")

features_prev = prev.groupby('SK_ID_CURR').agg(

    # ── Recurrencia del cliente ───────────────────────────
    # ¿Cuántas veces ha solicitado crédito antes?
    n_solicitudes_previas    = ('SK_ID_PREV', 'count'),

    # ¿Cuántas solicitudes previas fueron aprobadas?
    n_aprobadas_previas      = ('NAME_CONTRACT_STATUS',
                                lambda x: (x == 'Approved').sum()),

    # ¿Cuántas fueron rechazadas?
    n_rechazadas_previas     = ('NAME_CONTRACT_STATUS',
                                lambda x: (x == 'Refused').sum()),

    # ¿Cuántas fueron canceladas por el propio cliente?
    n_canceladas_previas     = ('NAME_CONTRACT_STATUS',
                                lambda x: (x == 'Canceled').sum()),

    # ── Montos históricos ─────────────────────────────────
    # ¿Qué montos solicitó y le aprobaron antes?
    amt_credito_prev_max     = ('AMT_CREDIT', 'max'),
    amt_credito_prev_mean    = ('AMT_CREDIT', 'mean'),
    amt_aplicacion_prev_mean = ('AMT_APPLICATION', 'mean'),

    # ── Última decisión ───────────────────────────────────
    # ¿Hace cuánto fue su solicitud más reciente?
    dias_ultima_solicitud    = ('DAYS_DECISION', 'max'),

).reset_index()

# ── Variable derivada: tasa de aprobación histórica ──────────
# ¿Qué proporción de sus solicitudes previas fueron aprobadas?
features_prev['tasa_aprobacion_historica'] = (
    features_prev['n_aprobadas_previas'] /
    features_prev['n_solicitudes_previas']
)

# ── Indicador: cliente nuevo vs recurrente ────────────────────
# 0 = cliente nuevo (sin historial previo)
# 1 = cliente recurrente (tiene al menos 1 solicitud previa)
features_prev['es_cliente_recurrente'] = (
    features_prev['n_solicitudes_previas'] > 0
).astype(int)

features_prev.columns.name = None

print(f"✅ Features de previous_application: {features_prev.shape}")
print(f"   Clientes únicos procesados: {features_prev.shape[0]:,}")
print()
print(features_prev.head())

Calculando features de previous_application...
✅ Features de previous_application: (338857, 11)
   Clientes únicos procesados: 338,857

   SK_ID_CURR  n_solicitudes_previas  n_aprobadas_previas  \
0      100001                      1                    1   
1      100002                      1                    1   
2      100003                      3                    3   
3      100004                      1                    1   
4      100005                      2                    1   

   n_rechazadas_previas  n_canceladas_previas  amt_credito_prev_max  \
0                     0                     0            23787.0000   
1                     0                     0           179055.0000   
2                     0                     0          1035882.0000   
3                     0                     0            20106.0000   
4                     0                     1            40153.5000   

   amt_credito_prev_mean  amt_aplicacion_prev_mean  dias_ultima_solici

In [20]:
# ============================================================
# CONSTRUCCIÓN DEL DATASET FINAL DE COBRANZA
# ============================================================
# Une los cuatro DataFrames usando SK_ID_CURR como llave.
# how='left' conserva todos los clientes de application_train
# aunque no tengan historial en los otros archivos.
#
# Clientes sin historial quedan con NaN — los trataremos
# en el notebook de modelado con imputación.
# ============================================================

print("Construyendo dataset final...")

# ── Merge de los cuatro DataFrames ───────────────────────────
df_collections = app.copy()   # Base: datos del solicitante

# Agregar historial de pagos por cuota
df_collections = df_collections.merge(
    features_inst, on='SK_ID_CURR', how='left'
)

# Agregar estado mensual por producto (DPD)
df_collections = df_collections.merge(
    features_pos, on='SK_ID_CURR', how='left'
)

# Agregar historial de solicitudes previas
df_collections = df_collections.merge(
    features_prev, on='SK_ID_CURR', how='left'
)

# ── Variables derivadas importantes ─────────────────────────

# Plazo estimado del crédito en meses
# AMT_CREDIT / AMT_ANNUITY = número aproximado de cuotas
df_collections['plazo_meses'] = (
    df_collections['AMT_CREDIT'] /
    df_collections['AMT_ANNUITY'].replace(0, np.nan)
).round(0)

# Carga financiera: qué proporción del ingreso mensual es la cuota
# AMT_ANNUITY es mensual, AMT_INCOME_TOTAL es anual → dividimos entre 12
df_collections['carga_financiera'] = (
    df_collections['AMT_ANNUITY'] /
    (df_collections['AMT_INCOME_TOTAL'] / 12)
).clip(upper=5)  # Limitar valores extremos

# Edad en años (DAYS_BIRTH es negativo)
df_collections['edad_anos'] = (
    -df_collections['DAYS_BIRTH'] / 365
).round(1)

# Antigüedad laboral en años (DAYS_EMPLOYED es negativo)
# Nota: valores positivos en DAYS_EMPLOYED son pensionados/desempleados
df_collections['antiguedad_laboral_anos'] = (
    -df_collections['DAYS_EMPLOYED'].clip(upper=0) / 365
).round(1)

# ── Resumen del dataset final ─────────────────────────────────
print(f"\n✅ Dataset de cobranza construido:")
print(f"   Filas (clientes)    : {df_collections.shape[0]:>8,}")
print(f"   Columnas (features) : {df_collections.shape[1]:>8,}")
print(f"\n📊 Cobertura por fuente:")
print(f"   Con historial installments : "
      f"{df_collections['max_dias_atraso'].notna().sum():>8,} "
      f"({df_collections['max_dias_atraso'].notna().mean():.1%})")
print(f"   Con historial POS_CASH     : "
      f"{df_collections['max_dpd'].notna().sum():>8,} "
      f"({df_collections['max_dpd'].notna().mean():.1%})")
print(f"   Con solicitudes previas    : "
      f"{df_collections['n_solicitudes_previas'].notna().sum():>8,} "
      f"({df_collections['n_solicitudes_previas'].notna().mean():.1%})")
print(f"\n🎯 Distribución TARGET:")
print(df_collections['TARGET'].value_counts(normalize=True)
      .rename({0: 'Pagó correctamente', 1: 'Incumplió'}))

# ── Guardar el dataset procesado ─────────────────────────────
output_path = f'{PROC}/df_collections.csv'
df_collections.to_csv(output_path, index=False)
print(f"\n💾 Dataset guardado en: {output_path}")

Construyendo dataset final...

✅ Dataset de cobranza construido:
   Filas (clientes)    :  307,511
   Columnas (features) :       54

📊 Cobertura por fuente:
   Con historial installments :  291,635 (94.8%)
   Con historial POS_CASH     :  289,444 (94.1%)
   Con solicitudes previas    :  291,057 (94.6%)

🎯 Distribución TARGET:
TARGET
Pagó correctamente   0.9193
Incumplió            0.0807
Name: proportion, dtype: float64

💾 Dataset guardado en: C:\Users\Marin\Documents\PROYECTO ML_OPS\credit-risk-scoring-ml\data\processed/df_collections.csv


In [21]:
# ============================================================
# CELDA 9 — CONSTRUCCIÓN DEL TARGET CON VENTANA TEMPORAL
# ============================================================
# Objetivo: construir el TARGET real del Early Warning System
# usando separación temporal del historial de pagos.
#
# Lógica:
# ┌─────────────────────────────────────────────────────────┐
# │  HISTORIAL COMPLETO DEL CLIENTE                         │
# │  ─────────────────────────────────────────────────────  │
# │  [── PERÍODO PASADO ──][── PERÍODO FUTURO ──]           │
# │       (FEATURES)            (TARGET)                    │
# │                      ↑                                  │
# │                 PUNTO DE CORTE                          │
# │               (percentil 50)                            │
# └─────────────────────────────────────────────────────────┘
#
# ⚠️  Data Leakage:
# Si calculamos features con TODO el historial incluyendo
# el futuro, el modelo "hace trampa" — ve la respuesta
# antes de predecirla. En producción esa info no existe.
#
# ⚠️  Clientes con historial insuficiente:
# Clientes con pocas cuotas casi nunca muestran mora —
# no por ser buenos pagadores sino por falta de tiempo.
# Se excluyen del EWS y se evalúan con el Agente de
# Originación (ya implementado).
# ============================================================

print("=" * 55)
print("CONSTRUCCIÓN DEL TARGET EWS")
print("=" * 55)
print(f"Umbral de mora     : > {UMBRAL_MORA_DIAS} días de atraso")
print(f"Mín. cuotas pasado : {MIN_CUOTAS_PASADAS} cuotas")
print(f"Mín. cuotas futuro : {MIN_CUOTAS_FUTURAS} cuotas")

# ── Paso 1: Definir punto de corte por cliente ───────────────
# El punto de corte es el percentil 50 del historial
# de cada cliente — el 50% más antiguo son las features
# y el 50% más reciente es donde observamos el TARGET

print("\n[1/6] Calculando punto de corte por cliente...")

punto_corte = (
    inst.groupby('SK_ID_CURR')['DAYS_INSTALMENT']
    .quantile(0.50)
    .reset_index()
)
punto_corte.columns = ['SK_ID_CURR', 'punto_corte']

print(f"      ✅ {punto_corte.shape[0]:,} clientes procesados")
print(f"      Punto de corte promedio: "
      f"{punto_corte['punto_corte'].mean():.0f} días")

# ── Paso 2: Separar historial en pasado y futuro ─────────────
# Unimos el punto de corte con el historial completo
# para poder filtrar por período

print("\n[2/6] Separando historial en pasado/futuro...")

inst_con_corte = inst.merge(
    punto_corte, on='SK_ID_CURR', how='left'
)

# PASADO: cuotas en o antes del punto de corte → FEATURES
inst_pasado = inst_con_corte[
    inst_con_corte['DAYS_INSTALMENT'] <=
    inst_con_corte['punto_corte']
].copy()

# FUTURO: cuotas después del punto de corte → TARGET
inst_futuro = inst_con_corte[
    inst_con_corte['DAYS_INSTALMENT'] >
    inst_con_corte['punto_corte']
].copy()

print(f"      ✅ Período pasado  : {inst_pasado.shape[0]:>10,} cuotas")
print(f"      ✅ Período futuro  : {inst_futuro.shape[0]:>10,} cuotas")

# ── Paso 3: Calcular TARGET desde el período futuro ──────────
# TARGET_EWS = 1 si el cliente tuvo mora > UMBRAL_MORA_DIAS
# en el período futuro

print("\n[3/6] Calculando TARGET_EWS desde período futuro...")

target_ews = inst_futuro.groupby('SK_ID_CURR').agg(
    # Peor atraso observado en el futuro
    max_atraso_futuro = ('dias_atraso', 'max'),
    # Cuántas cuotas tiene en el futuro
    # (necesario para el filtro de historial mínimo)
    n_cuotas_futuras  = ('dias_atraso', 'count'),
).reset_index()

# TARGET binario: mora si atraso supera el umbral configurado
target_ews['TARGET_EWS'] = (
    target_ews['max_atraso_futuro'] > UMBRAL_MORA_DIAS
).astype(int)

print(f"      ✅ TARGET calculado para {target_ews.shape[0]:,} clientes")
print(f"      Tasa de mora bruta: {target_ews['TARGET_EWS'].mean():.1%}")

# ── Paso 4: Aplicar filtro de historial mínimo ───────────────
# Excluir clientes con historial insuficiente para evitar
# falsos negativos por falta de tiempo de observación

print("\n[4/6] Aplicando filtro de historial mínimo...")

# Contar cuotas en el período pasado por cliente
n_cuotas_pasado = (
    inst_pasado.groupby('SK_ID_CURR')
    .size()
    .reset_index()
)
n_cuotas_pasado.columns = ['SK_ID_CURR', 'n_cuotas_pasado']

# Agregar conteo al target
target_ews = target_ews.merge(
    n_cuotas_pasado, on='SK_ID_CURR', how='left'
)

# Registrar estado antes del filtro
total_antes = len(target_ews)
mora_antes  = target_ews['TARGET_EWS'].mean()

# Aplicar filtros definidos en la configuración
# Un cliente debe tener suficientes cuotas en AMBOS períodos
mask_suficiente = (
    (target_ews['n_cuotas_futuras'] >= MIN_CUOTAS_FUTURAS) &
    (target_ews['n_cuotas_pasado']  >= MIN_CUOTAS_PASADAS)
)

# Clientes excluidos por historial insuficiente
excluidos = target_ews[~mask_suficiente].copy()

# Dataset filtrado listo para modelado
target_ews_filtrado = target_ews[mask_suficiente].copy()

print(f"\n      📊 Resultado del filtro:")
print(f"      Clientes antes del filtro   : {total_antes:>8,}")
print(f"      Excluidos — pocas cuotas fut: "
      f"{(target_ews['n_cuotas_futuras'] < MIN_CUOTAS_FUTURAS).sum():>8,}")
print(f"      Excluidos — pocas cuotas pas: "
      f"{(target_ews['n_cuotas_pasado'] < MIN_CUOTAS_PASADAS).sum():>8,}")
print(f"      Clientes para modelado      : "
      f"{len(target_ews_filtrado):>8,} "
      f"({len(target_ews_filtrado)/total_antes:.1%})")
print(f"\n      Tasa de mora antes del filtro : {mora_antes:.1%}")
print(f"      Tasa de mora después del filtro: "
      f"{target_ews_filtrado['TARGET_EWS'].mean():.1%}")
print(f"\n      ℹ️  Los {len(excluidos):,} clientes excluidos se")
print(f"         evalúan con el Agente de Originación")

# ── Paso 5: Recalcular features con período pasado ───────────
# IMPORTANTE: recalculamos con inst_pasado únicamente
# para garantizar que no hay data leakage

print("\n[5/6] Recalculando features con período pasado "
      "(sin data leakage)...")

features_inst_ews = inst_pasado.groupby('SK_ID_CURR').agg(

    # ── Intensidad del atraso histórico ──────────────────
    max_dias_atraso      = ('dias_atraso', 'max'),
    mean_dias_atraso     = ('dias_atraso', 'mean'),
    std_dias_atraso      = ('dias_atraso', 'std'),

    # ── Frecuencia del atraso ────────────────────────────
    pct_cuotas_atrasadas = ('dias_atraso',
                            lambda x: (x > 0).mean()),
    pct_cuotas_30dias    = ('dias_atraso',
                            lambda x: (x > 30).mean()),
    pct_cuotas_60dias    = ('dias_atraso',
                            lambda x: (x > 60).mean()),
    pct_cuotas_90dias    = ('dias_atraso',
                            lambda x: (x > 90).mean()),

    # ── Comportamiento de pago ───────────────────────────
    pct_pagos_parciales  = ('pago_parcial', 'mean'),
    pct_pagos_completos  = ('pago_completo', 'mean'),

    # ── Mora según umbral de configuración ───────────────
    n_cuotas_en_mora     = (f'mora_{UMBRAL_MORA_DIAS}d', 'sum'),
    pct_cuotas_en_mora   = (f'mora_{UMBRAL_MORA_DIAS}d', 'mean'),

    # ── Volumen del historial ────────────────────────────
    n_cuotas_total       = ('NUM_INSTALMENT_NUMBER', 'count'),
    n_cuotas_recientes   = ('DAYS_INSTALMENT',
                            lambda x: (x > -365).sum()),

).reset_index()

features_inst_ews.columns.name = None
print(f"      ✅ Features recalculadas: {features_inst_ews.shape}")

# ── Paso 6: Construir dataset EWS final ──────────────────────
# Unimos features del pasado (sin leakage) + TARGET del futuro

print("\n[6/6] Construyendo dataset EWS final...")

df_ews = app.copy()

# Features de comportamiento (período pasado — sin leakage)
df_ews = df_ews.merge(features_inst_ews,
                       on='SK_ID_CURR', how='left')
df_ews = df_ews.merge(features_pos,
                       on='SK_ID_CURR', how='left')
df_ews = df_ews.merge(features_prev,
                       on='SK_ID_CURR', how='left')

# TARGET real del EWS (período futuro)
df_ews = df_ews.merge(
    target_ews_filtrado[['SK_ID_CURR', 'TARGET_EWS']],
    on='SK_ID_CURR', how='left'
)

# ── Variables derivadas ──────────────────────────────────────

# Plazo estimado en meses (monto / cuota mensual)
df_ews['plazo_meses'] = (
    df_ews['AMT_CREDIT'] /
    df_ews['AMT_ANNUITY'].replace(0, np.nan)
).round(0)

# Carga financiera: proporción del ingreso mensual que es cuota
# AMT_ANNUITY es mensual, AMT_INCOME_TOTAL es anual → /12
df_ews['carga_financiera'] = (
    df_ews['AMT_ANNUITY'] /
    (df_ews['AMT_INCOME_TOTAL'] / 12)
).clip(upper=5)  # Limitar valores extremos atípicos

# Edad en años (DAYS_BIRTH es negativo)
df_ews['edad_anos'] = (
    -df_ews['DAYS_BIRTH'] / 365
).round(1)

# Antigüedad laboral en años
# Nota: DAYS_EMPLOYED positivo = pensionado/desempleado → clip en 0
df_ews['antiguedad_laboral_anos'] = (
    -df_ews['DAYS_EMPLOYED'].clip(upper=0) / 365
).round(1)

# ── Solo clientes con TARGET_EWS definido ────────────────────
# Excluimos clientes sin historial suficiente del modelado
df_ews_modelo = df_ews.dropna(subset=['TARGET_EWS']).copy()

# ── Reporte final ─────────────────────────────────────────────
print(f"\n{'='*55}")
print(f"DATASET EWS FINAL")
print(f"{'='*55}")
print(f"Clientes totales en app_train    : "
      f"{df_ews.shape[0]:>8,}")
print(f"Clientes para modelado EWS       : "
      f"{df_ews_modelo.shape[0]:>8,} "
      f"({df_ews_modelo.shape[0]/df_ews.shape[0]:.1%})")
print(f"Features disponibles             : "
      f"{df_ews_modelo.shape[1]:>8,}")

print(f"\n🎯 Distribución TARGET_EWS:")
dist = df_ews_modelo['TARGET_EWS'].value_counts(normalize=True)
print(f"   Sin mora futura (0) : "
      f"{dist.get(0.0, 0):.1%}")
print(f"   Con mora futura (1) : "
      f"{dist.get(1.0, 0):.1%}")

print(f"\n📊 Comparación de targets:")
print(f"   TARGET original (incumplimiento total)  : "
      f"{df_ews_modelo['TARGET'].mean():.1%}")
print(f"   TARGET_EWS (mora futura > {UMBRAL_MORA_DIAS} días)      : "
      f"{df_ews_modelo['TARGET_EWS'].mean():.1%}")
print(f"\n   ℹ️  La diferencia entre ambos targets es normal —")
print(f"   TARGET original incluye incumplimientos")
print(f"   tardíos que el EWS no puede predecir aún.")

# ── Guardar dataset EWS ──────────────────────────────────────
output_ews = f'{PROC}/df_ews.csv'
df_ews_modelo.to_csv(output_ews, index=False)
print(f"\n💾 Dataset EWS guardado en: {output_ews}")

CONSTRUCCIÓN DEL TARGET EWS
Umbral de mora     : > 5 días de atraso
Mín. cuotas pasado : 6 cuotas
Mín. cuotas futuro : 3 cuotas

[1/6] Calculando punto de corte por cliente...
      ✅ 339,587 clientes procesados
      Punto de corte promedio: -881 días

[2/6] Separando historial en pasado/futuro...
      ✅ Período pasado  :  6,905,068 cuotas
      ✅ Período futuro  :  6,700,333 cuotas

[3/6] Calculando TARGET_EWS desde período futuro...
      ✅ TARGET calculado para 338,580 clientes
      Tasa de mora bruta: 18.2%

[4/6] Aplicando filtro de historial mínimo...

      📊 Resultado del filtro:
      Clientes antes del filtro   :  338,580
      Excluidos — pocas cuotas fut:   17,183
      Excluidos — pocas cuotas pas:   65,177
      Clientes para modelado      :  273,391 (80.7%)

      Tasa de mora antes del filtro : 18.2%
      Tasa de mora después del filtro: 21.6%

      ℹ️  Los 65,189 clientes excluidos se
         evalúan con el Agente de Originación

[5/6] Recalculando features con p

In [13]:
# ============================================================
# CELDA 10 — DIAGNÓSTICO DEL DESBALANCE DE CLASES
# ============================================================
# Objetivo: entender el impacto del desbalance y evaluar
# qué ajuste tiene más sentido antes de modelar.
# ============================================================

print("=" * 55)
print("DIAGNÓSTICO DEL DESBALANCE DE CLASES")
print("=" * 55)

# ── 1. Impacto de diferentes umbrales de mora ────────────────
print("\n📊 Tasa de mora según diferentes umbrales:")
print(f"{'Umbral':>10} {'Tasa mora':>12} {'N clientes mora':>18}")
print("-" * 45)

for umbral in [0, 5, 10, 15, 20, 30, 45, 60, 90]:
    # Recalcular TARGET con este umbral
    target_temp = inst_futuro.groupby('SK_ID_CURR').agg(
        max_atraso = ('dias_atraso', 'max')
    ).reset_index()
    target_temp['target_temp'] = (
        target_temp['max_atraso'] > umbral
    ).astype(int)

    # Solo clientes en nuestro dataset de modelado
    target_temp = target_temp[
        target_temp['SK_ID_CURR'].isin(
            df_ews_modelo['SK_ID_CURR']
        )
    ]

    tasa = target_temp['target_temp'].mean()
    n_mora = target_temp['target_temp'].sum()
    print(f"  > {umbral:>4} días  {tasa:>10.1%}   {n_mora:>14,}")

# ── 2. Impacto del punto de corte ────────────────────────────
print("\n📊 Tasa de mora según punto de corte temporal:")
print(f"{'Percentil':>12} {'Tasa mora':>12} {'N modelado':>12}")
print("-" * 40)

for pct in [40, 50, 60, 70, 80]:
    corte_temp = (
        inst.groupby('SK_ID_CURR')['DAYS_INSTALMENT']
        .quantile(pct/100)
        .reset_index()
    )
    corte_temp.columns = ['SK_ID_CURR', 'corte']

    inst_fut_temp = inst.merge(
        corte_temp, on='SK_ID_CURR', how='left'
    )
    inst_fut_temp = inst_fut_temp[
        inst_fut_temp['DAYS_INSTALMENT'] >
        inst_fut_temp['corte']
    ]

    target_temp = inst_fut_temp.groupby('SK_ID_CURR').agg(
        max_atraso     = ('dias_atraso', 'max'),
        n_cuotas_fut   = ('dias_atraso', 'count')
    ).reset_index()

    target_temp = target_temp[
        target_temp['n_cuotas_fut'] >= MIN_CUOTAS_FUTURAS
    ]
    target_temp['target_temp'] = (
        target_temp['max_atraso'] > UMBRAL_MORA_DIAS
    ).astype(int)

    tasa = target_temp['target_temp'].mean()
    n = len(target_temp)
    print(f"  Percentil {pct:>3}  {tasa:>10.1%}  {n:>12,}")

# ── 3. Resumen del problema ──────────────────────────────────
print(f"\n{'='*55}")
print("RESUMEN DEL PROBLEMA")
print(f"{'='*55}")
print(f"Tasa actual de mora EWS : 1.8%")
print(f"Ratio desbalance        : ~54 buenos por 1 malo")
print(f"Tasa Agente Originación : 8.1%")
print(f"Ratio desbalance orig.  : ~11 buenos por 1 malo")
print(f"\nUn modelo que prediga siempre 0 tendría:")
print(f"  Accuracy : 98.2% (inútil para cobranza)")
print(f"  Recall   :  0.0% (no detecta ningún cliente en mora)")
print(f"\nLa métrica correcta para este modelo NO es accuracy")
print(f"sino RECALL y AUC-ROC — ¿qué % de morosos detecta?")

DIAGNÓSTICO DEL DESBALANCE DE CLASES

📊 Tasa de mora según diferentes umbrales:
    Umbral    Tasa mora    N clientes mora
---------------------------------------------
  >    0 días       34.5%           86,937
  >    5 días       13.7%           34,521
  >   10 días        6.6%           16,651
  >   15 días        4.4%           11,028
  >   20 días        3.4%            8,566
  >   30 días        1.8%            4,531
  >   45 días        1.0%            2,447
  >   60 días        0.8%            2,074
  >   90 días        0.6%            1,578

📊 Tasa de mora según punto de corte temporal:
   Percentil    Tasa mora   N modelado
----------------------------------------
  Percentil  40        3.5%       327,876
  Percentil  50        2.9%       321,397
  Percentil  60        2.4%       307,399
  Percentil  70        1.7%       299,982
  Percentil  80        1.1%       260,746

RESUMEN DEL PROBLEMA
Tasa actual de mora EWS : 1.8%
Ratio desbalance        : ~54 buenos por 1 malo
Tasa A